[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xiptos/is_notes/blob/main/autoencoder_cnn.ipynb)

# Convolution Autoencoders (CAE)

The encoder and the decoder use convolutional layers, which are better suited to images than fully connected layers. The encoder reduces each 28x28 image to 8 feature maps of 7x7 (392 values); the decoder uses transposed convolutions to go back to 28x28.

Import the libraries

In [ ]:
import torch
from torchvision import datasets
from torchvision import transforms
import matplotlib.pyplot as plt
import numpy as np
import os
from torchvision.utils import save_image

from os.path import exists

Utility functions, to store the temporary images and to convert matrix shape.

In [ ]:
os.makedirs('./cae_img', exist_ok=True)


def to_img(x):
    x = x.clamp(0, 1)
    x = x.view(x.size(0), 1, 28, 28)
    return x

## Dataset

In [ ]:
num_epochs = 20
batch_size = 128
learning_rate = 1e-3

# Transforms images to a PyTorch Tensor, with values in [0, 1]
# (the decoder ends with a sigmoid, which also outputs values in [0, 1])
img_transform = transforms.ToTensor()

# Download the MNIST Dataset
dataset = datasets.MNIST(root="./data", train=True, download=True, transform=img_transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=img_transform)

# DataLoader is used to load the dataset
# for training
loader = torch.utils.data.DataLoader(dataset=dataset, batch_size=batch_size, shuffle=True)

## Model

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"Using device: {device}")

In [ ]:
# Creating a PyTorch class
# 1x28x28 ==> 8x7x7 ==> 1x28x28
class AE(torch.nn.Module):
    def __init__(self):
        super().__init__()

        self.encoder = torch.nn.Sequential(
            torch.nn.Conv2d(1, 16, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2),
            torch.nn.Conv2d(16, 8, kernel_size=3, stride=1, padding=1),
            torch.nn.ReLU(),
            torch.nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.decoder = torch.nn.Sequential(
            torch.nn.ConvTranspose2d(8, 16,
                               kernel_size=3,
                               stride=2,
                               padding=1,
                               output_padding=1),
            torch.nn.ReLU(),
            torch.nn.ConvTranspose2d(16, 1,
                               kernel_size=3,
                               stride=2,
                               padding=1,
                               output_padding=1),
            torch.nn.Sigmoid()
        )
        
    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return encoded, decoded

In [ ]:
# Model Initialization
model = AE()
#moving to gpu
model.to(device)

# Validation using MSE Loss function
loss_function = torch.nn.MSELoss()

# Using an Adam Optimizer
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate, weight_decay=1e-5)

## Training/Loading

If the model was already trained and saved, it is loaded from the file. Otherwise, it is trained (delete the file to train it again).

In [ ]:
model_file = "autoencoder_cnn.pth"
img_dir = "cae_img"

In [ ]:
if exists(model_file):
    model.load_state_dict(torch.load(model_file, map_location=device, weights_only=True))
    model.eval()
else:
    losses = []
    model.train()
    for epoch in range(num_epochs):
        for data in loader:
            image, label = data

            image = image.to(device)

            # Output of Autoencoder
            _, reconstructed = model(image)

            # Calculating the loss function
            loss = loss_function(reconstructed, image)

            # The gradients are set to zero,
            # the gradient is computed and stored.
            # .step() performs parameter update
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            # Storing the losses in a list for plotting
            losses.append(loss.item())
        print('epoch [{}/{}], loss:{:.4f}'.format(epoch + 1, num_epochs, loss.item()))
        if epoch % 5 == 0:
            pic = to_img(reconstructed.cpu().data)
            save_image(pic, './{}/image_{}.png'.format(img_dir, epoch))

    torch.save(model.state_dict(), model_file)
    model.eval()

    plt.xlabel('Iterations')
    plt.ylabel('Loss')
    plt.plot(losses)
    plt.show()

## Display sample images

Test images (top) and their reconstructions (bottom).

In [ ]:
import random
from mpl_toolkits.axes_grid1 import ImageGrid


def show_reconstructions(dataset, filename=None):
    """Shows 10 random images (top row) and their reconstructions (bottom row)."""
    model.eval()
    fig = plt.figure(figsize=(20., 5.))
    grid = ImageGrid(fig, 111,  # similar to subplot(111)
                     nrows_ncols=(2, 10),  # creates 2x10 grid of axes
                     axes_pad=0.1,  # pad between axes in inch.
                     )

    for i in range(10):
        image, _ = dataset[random.randrange(len(dataset))]

        img = image.unsqueeze(0).to(device)
        with torch.no_grad():
            _, reconstructed = model(img)
        item = reconstructed.reshape(-1, 28, 28).cpu().numpy()

        # Iterating over the grid returns the Axes.
        grid[i].imshow(image[0], cmap='gray')
        grid[i].axis('off')
        grid[10+i].imshow(item[0], cmap='gray')
        grid[10+i].axis('off')

    if filename:
        plt.savefig(filename, transparent=True)
    plt.show()


show_reconstructions(test_dataset, "cae_nums.png")